We make one simple LLM call and print the reply.


In [ ]:
import os
import json
import urllib.request

API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

def ask(messages):
    """messages: string OR list of {role, content} dicts."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not API_KEY:
        raise SystemExit("Set OPENAI_API_KEY and run again.")
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"]

reply = ask("Remember this secret: the codeword is ORBIT-77.")
print(reply)


Now ask the model what it just said.


In [ ]:
# New call — do not pass the previous cell's history
reply2 = ask("What did you just say?")
print(reply2)


To make the model "remember", build a history: a list of `role` / `content` messages, with the latest message at the end.


In [ ]:
history = [
    {"role": "user", "content": "Remember this secret: the codeword is ORBIT-77."},
    {"role": "assistant", "content": reply},
    {"role": "user", "content": "What is the codeword?"},
]

reply3 = ask(history)
print(reply3)


If the history grows too large, the context window overflows and the provider returns an error. Let's force that.


In [ ]:
# Grow the history until the API fails (e.g. 400 context length exceeded)
big_history = [
    {"role": "user", "content": "Remember this secret: the codeword is ORBIT-77."},
    {"role": "assistant", "content": reply},
]

blob = "x" * 50_000  # large text only to fill the context

try:
    while True:
        big_history.append({"role": "user", "content": blob})
        big_history.append({"role": "assistant", "content": "ok"})
        print(f"messages in history: {len(big_history)}")
        ask(big_history)
except Exception as e:
    print("Provider error:")
    print(e)
